# ChuckleNet Scale500: GPU Pipeline

**Goal:** Scale to 500+ labeled videos using GPU acceleration.

**Pipeline:**
1. Mount Drive + install dependencies
2. Collect video IDs from StandUp4AI partition
3. Download audio (yt-dlp, ~10min per 50 videos)
4. Extract WavLM embeddings (GPU, ~5min per 50 videos)
5. Extract prosody features (CPU, ~3min per 50 videos)
6. Pseudo-label with fusion model (F1=0.975)
7. Retrain fusion model on expanded data
8. Evaluate on held-out comedians

**Runtime:** ~2-3 hours for 300 videos (GPU)
**GPU:** Required for WavLM extraction


In [ ]:
# Cell 1: Setup + Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, json, subprocess, warnings
warnings.filterwarnings('ignore')

BASE = '/content/drive/MyDrive/standup4ai'
SCALE_DIR = f'{BASE}/scale500'
os.makedirs(SCALE_DIR, exist_ok=True)

# Check space
result = subprocess.run(['df', '-h', BASE], capture_output=True, text=True)
print(result.stdout)

# Install
subprocess.run(['pip', 'install', 'yt-dlp', '-q'], capture_output=True)
subprocess.run(['pip', 'install', 'librosa', '-q'], capture_output=True)
print('Done')

In [ ]:
# Cell 2: Get video candidates from StandUp4AI partition
import pandas as pd
import glob

partition = pd.read_csv(f'{BASE}/standup4ai_partition.csv')
print(f'Total in partition: {len(partition)}')
print(partition['part'].value_counts())

train_vids = set(partition[partition['part'] == 'train']['fn'].tolist())

# Check existing audio
existing = set()
for d in [f'{BASE}/audio', f'{BASE}/audio_1000']:
    if os.path.exists(d):
        existing.update(f.rsplit('.', 1)[0] for f in os.listdir(d))

need = sorted(list(train_vids - existing))
print(f'Existing: {len(existing)}, Need: {len(need)}')

# Target 300 for this run
target = need[:300]
with open(f'{SCALE_DIR}/candidates.json', 'w') as f:
    json.dump(target, f)
print(f'Target this run: {len(target)}')

In [ ]:
# Cell 3: Download audio (yt-dlp)
AUDIO_OUT = f'{SCALE_DIR}/audio'
os.makedirs(AUDIO_OUT, exist_ok=True)

def download_batch(vids, batch_size=10):
    for i, vid in enumerate(vids):
        out = f'{AUDIO_OUT}/{vid}.m4a'
        if os.path.exists(out):
            print(f'  {i+1}. {vid} (exists)')
            continue
        cmd = ['yt-dlp', '-f', 'bestaudio[ext=m4a]',
               '-o', f'{out}.%(ext)s',
               f'https://www.youtube.com/watch?v={vid}',
               '--no-playlist', '--quiet', '--socket-timeout', '30']
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        status = 'OK' if r.returncode == 0 else 'FAIL'
        # Rename
        for ext in ['m4a', 'webm', 'mp4']:
            tmp = f'{out}.{ext}'
            if os.path.exists(tmp) and tmp != out:
                os.rename(tmp, out)
        print(f'  {i+1}. {vid} [{status}]')

# Resume from checkpoint
ckpt = f'{SCALE_DIR}/download_ckpt.json'
start_idx = 0
if os.path.exists(ckpt):
    with open(ckpt) as f:
        done = set(json.load(f).get('done', []))
    target = [v for v in target if v not in done]
    print(f'Resuming: {len(target)} remaining')

print(f'Downloading {len(target)} videos...')
download_batch(target)
n = len([f for f in os.listdir(AUDIO_OUT) if f.endswith('.m4a')])
print(f'\nTotal audio files: {n}')

In [ ]:
# Cell 4: Load WavLM (GPU)
import torch
from transformers import AutoModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

print('Loading WavLM...')
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.to(device)
wavlm.eval()
print('WavLM ready')

In [ ]:
# Cell 5: Feature extraction functions
import librosa
import numpy as np
import soundfile as sf

SR_WAVLM = 16000
SR_PROSODY = 22050
SEG_LEN = 5.0   # 5-second segments
STRIDE = 2.5    # 50% overlap

def extract_wavlm_segment(chunk):
    """GPU: extract WavLM 768-dim embedding from 5s chunk."""
    chunk_t = torch.tensor(chunk).unsqueeze(0).to(device)
    with torch.no_grad():
        rep = wavlm(chunk_t).last_hidden_state
    return rep.mean(dim=1).squeeze().cpu().numpy()

def extract_prosody_segment(y, sr):
    """CPU: extract 15-dim prosody features."""
    if len(y) < 0.5 * sr: return None
    hop = 512
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0]
    sc = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]
    feats = [np.mean(sc), np.std(sc), np.mean(rms), np.std(rms), np.max(rms),
             np.mean(zcr), np.std(zcr)]
    if len(y) / sr >= 0.5:
        try:
            f0, voiced, _ = librosa.pyin(y, fmin=80, fmax=500, sr=sr)
            f0 = np.nan_to_num(f0, nan=0)
            voiced = np.nan_to_num(voiced, nan=0)
            feats += [np.mean(f0), np.std(f0), np.mean(voiced)]
        except:
            feats += [0, 0, 0]
    else:
        feats += [0, 0, 0]
    # Pad/truncate to 15
    feats = feats[:15] + [0] * max(0, 15 - len(feats))
    return np.array(feats, dtype=np.float32)

def extract_video(audio_path):
    """Extract WavLM + prosody for all segments in a video."""
    try:
        # Load at both sample rates
        y16, _ = librosa.load(audio_path, sr=SR_WAVLM, mono=True)
        y22, _ = librosa.load(audio_path, sr=SR_PROSODY, mono=True)
    except:
        return None
    
    wavlm_feats, prosody_feats = [], []
    max_dur = min(len(y16)/SR_WAVLM, 300)  # max 5 min per video
    
    for t in np.arange(0, max_dur, STRIDE):
        # WavLM chunk
        s16, e16 = int(t*SR_WAVLM), int((t+SEG_LEN)*SR_WAVLM)
        if e16 > len(y16): break
        chunk = y16[s16:e16]
        if len(chunk) < 0.5*SR_WAVLM: continue
        if len(chunk) < SEG_LEN*SR_WAVLM:
            chunk = np.pad(chunk, (0, int(SEG_LEN*SR_WAVLM) - len(chunk)))
        wavlm_feats.append(extract_wavlm_segment(chunk))
        
        # Prosody chunk
        s22, e22 = int(t*SR_PROSODY), int((t+SEG_LEN)*SR_PROSODY)
        chunk22 = y22[s22:e22] if e22 <= len(y22) else y22[s22:]
        pf = extract_prosody_segment(chunk22, SR_PROSODY)
        prosody_feats.append(pf if pf is not None else np.zeros(15, dtype=np.float32))
    
    if not wavlm_feats: return None
    
    n = min(len(wavlm_feats), len(prosody_feats))
    wavlm_feats = np.array(wavlm_feats[:n])
    prosody_feats = np.array(prosody_feats[:n])
    combined = np.concatenate([wavlm_feats, prosody_feats], axis=1)  # (n, 783)
    return combined

print('Feature extractors ready')

In [ ]:
# Cell 6: Process all audio files
from tqdm import tqdm
import time

EMB_DIR = f'{SCALE_DIR}/embeddings'
os.makedirs(EMB_DIR, exist_ok=True)

AUDIO_DIR = f'{SCALE_DIR}/audio'
audio_files = sorted([f for f in os.listdir(AUDIO_DIR) if f.endswith('.m4a')])
print(f'Audio files: {len(audio_files)}')

# Checkpoint
ckpt_file = f'{SCALE_DIR}/extract_ckpt.json'
done = set()
if os.path.exists(ckpt_file):
    with open(ckpt_file) as f:
        done = set(json.load(f).get('done', []))
print(f'Already done: {len(done)}')

all_data = []
for fname in tqdm(audio_files):
    vid = fname.rsplit('.', 1)[0]
    if vid in done: continue
    
    emb = extract_video(f'{AUDIO_DIR}/{fname}')
    if emb is not None and len(emb) > 0:
        np.save(f'{EMB_DIR}/{vid}.npy', emb)
        all_data.append({'vid': vid, 'n_segs': len(emb)})
    
    done.add(vid)
    if len(done) % 20 == 0:
        with open(ckpt_file, 'w') as f:
            json.dump({'done': list(done)}, f)

with open(ckpt_file, 'w') as f:
    json.dump({'done': list(done)}, f)

print(f'\nExtracted: {len(all_data)} videos')

In [ ]:
# Cell 7: Pseudo-label new segments — USE THE FUSION MODEL (F1=0.975)
# CRITICAL: Never pseudo-label with a broken model (lessons from past failures)
# past failure: top200_prosody_model.pt saturated because pos_weight=5.0
# lesson: only pseudo-label with a model that actually works

import torch
import numpy as np

# Load the fusion model (F1=0.975) for pseudo-labeling
FUSION_MODEL = f'{BASE}/models/fusion_model.pt'
FUSION_BACKUP = f'{BASE}/experiments/best_fusion_model.pt'

model_to_use = FUSION_MODEL if os.path.exists(FUSION_MODEL) else FUSION_BACKUP
print(f'Using model: {model_to_use}')

# Try to load fusion model for pseudo-labeling
class FusionMLP(nn.Module):
    def __init__(self, input_dim=783):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.BatchNorm1d(512), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.3),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

pseudo_model = None
if os.path.exists(model_to_use):
    try:
        pseudo_model = FusionMLP(input_dim=783)
        state = torch.load(model_to_use, map_location='cpu')
        pseudo_model.load_state_dict(state, strict=False)
        pseudo_model.eval()
        print('Fusion model loaded for pseudo-labeling')
    except Exception as e:
        print(f'Could not load fusion model: {e}')
        pseudo_model = None

# Load existing labeled data
EXISTING_NPZ = f'{BASE}/wavlm_prosody_expanded.npz'
X_existing, y_existing = None, None
if os.path.exists(EXISTING_NPZ):
    d = np.load(EXISTING_NPZ)
    X_existing, y_existing = d['X'], d['y']
    print(f'Existing labeled: {len(y_existing)} segs, {y_existing.sum()} pos ({100*y_existing.mean():.1f}%)')

# Load pseudo-labeled new data
EMB_DIR = f'{SCALE_DIR}/embeddings'
emb_files = sorted([f for f in os.listdir(EMB_DIR) if f.endswith('.npy')])
X_new_list, y_new_list, vids_new = [], [], []

for f in emb_files:
    vid = f.rsplit('.', 1)[0]
    emb = np.load(f'{EMB_DIR}/{f}')  # (n_segs, 783)
    X_new_list.append(emb)
    vids_new.extend([vid] * len(emb))

X_new = np.vstack(X_new_list) if X_new_list else None

# Pseudo-label using fusion model (NOT energy threshold!)
if X_new is not None and pseudo_model is not None:
    with torch.no_grad():
        probs = pseudo_model(torch.tensor(X_new, dtype=torch.float32)).numpy().squeeze()
    y_new = (probs >= 0.5).astype(int)
    pos_rate = y_new.mean()
    print(f'Pseudo-labeled ({len(y_new)} segs): {y_new.sum()} pos ({100*pos_rate:.1f}%)')
    print(f'Prob dist: min={probs.min():.4f}, max={probs.max():.4f}, mean={probs.mean():.4f}')
    
    # CRITICAL CHECK: reject if positive rate < 15% (sparsity threshold from historical lessons)
    if pos_rate < 0.15:
        print(f'WARNING: Positive rate {100*pos_rate:.1f}% < 15% threshold!')
        print(f'These pseudo-labels are too sparse — will cause model collapse.')
        print(f'Options: (1) lower threshold, (2) use only high-confidence, (3) skip pseudo-labels')
        # Fall back: use top 30% by probability as positive
        threshold = np.percentile(probs, 70)
        y_new = (probs >= threshold).astype(int)
        print(f'Fallback: using top 30% as positive. New pos rate: {100*y_new.mean():.1f}%')
elif X_new is not None:
    # No fusion model — use energy + diversity heuristic
    prosody_slice = X_new[:, 768:]  # last 15 dims = prosody
    rms_proxy = prosody_slice[:, 2]  # mean RMS
    energy_score = (rms_proxy - rms_proxy.min()) / (rms_proxy.max() - rms_proxy.min() + 1e-8)
    
    # Multi-signal: combine RMS + ZCR + spectral flatness
    zcr = prosody_slice[:, 5]  # ZCR
    flatness = prosody_slice[:, 13]  # spectral flatness
    zcr_norm = (zcr - zcr.min()) / (zcr.max() - zcr.min() + 1e-8)
    flatness_norm = (flatness - flatness.min()) / (flatness.max() - flatness.min() + 1e-8)
    
    # Laughter: high RMS + moderate ZCR + low flatness (tonal vs noise)
    laugh_score = 0.5 * energy_score + 0.3 * zcr_norm + 0.2 * (1 - flatness_norm)
    threshold = np.percentile(laugh_score, 70)
    y_new = (laugh_score >= threshold).astype(int)
    pos_rate = y_new.mean()
    print(f'Fallback pseudo-label: {len(y_new)} segs, {y_new.sum()} pos ({100*pos_rate:.1f}%)')

# Combine
if X_existing is not None and X_new is not None:
    X_all = np.vstack([X_existing, X_new])
    y_all = np.concatenate([y_existing, y_new])
    vids_all = ['existing'] * len(y_existing) + vids_new
elif X_new is not None:
    X_all, y_all, vids_all = X_new, y_new, vids_new
elif X_existing is not None:
    X_all, y_all, vids_all = X_existing, y_existing, ['existing'] * len(y_existing)
else:
    print('ERROR: No data available')
    X_all = y_all = vids_all = None

if y_all is not None:
    print(f'Total: {len(y_all)} segs, {y_all.sum()} pos ({100*y_all.mean():.1f}%)')


In [ ]:
# Cell 8: Retrain fusion model with GroupKFold
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score

class FusionMLP(nn.Module):
    def __init__(self, input_dim=783):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.BatchNorm1d(512), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.3),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

if X_all is None:
    print('No data to train')
else:
    groups = np.array(vids_all)
    unique_vids = list(set(vids_all))
    n_vids = len(unique_vids)
    print(f'Training on {len(y_all)} segments from {n_vids} videos')

    gkf = GroupKFold(n_splits=min(5, n_vids))
    models, scalers, fold_f1s = [], [], []

    for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_all, y_all, groups)):
        print(f'\nFold {fold+1}')
        Xtr, Xte = X_all[tr_idx], X_all[te_idx]
        ytr, yte = y_all[tr_idx], y_all[te_idx]

        scaler = StandardScaler()
        Xtr_s = scaler.fit_transform(Xtr)
        Xte_s = scaler.transform(Xte)

        model = FusionMLP(input_dim=783)
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
        criterion = nn.BCELoss()

        Xtr_t = torch.tensor(Xtr_s, dtype=torch.float32)
        ytr_t = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1)

        best_f1, patience, no_imp = 0, 5, 0
        for epoch in range(50):
            model.train()
            for i in range(0, len(Xtr_t), 256):
                bx = Xtr_t[i:i+256]
                by = ytr_t[i:i+256]
                opt.zero_grad()
                loss = criterion(model(bx), by)
                loss.backward()
                opt.step()

            model.eval()
            with torch.no_grad():
                preds = model(torch.tensor(Xte_s, dtype=torch.float32)).numpy().squeeze()
            f = f1_score(yte, (preds >= 0.5).astype(int))
            if f > best_f1:
                best_f1 = f
                no_imp = 0
            else:
                no_imp += 1
                if no_imp >= patience:
                    break

        model.eval()

        # SATURATION CHECK: Historical lesson — model can collapse to all 1.0 or all 0.0
        # If this happens, the training failed and we need to check data quality
        with torch.no_grad():
            sample_probs = model(torch.tensor(Xte_s[:100], dtype=torch.float32)).numpy().squeeze()
        prob_std = sample_probs.std()
        if prob_std < 0.01:
            print(f'  WARNING: Model appears saturated! prob_std={prob_std:.6f}')
            print(f'  Sample probs: min={sample_probs.min():.4f}, max={sample_probs.max():.4f}')
            print(f'  This means the model is predicting the same class for everything.')
            print(f'  Check: (1) positive rate in training data, (2) pos_weight setting')
        
        with torch.no_grad():
            preds = model(torch.tensor(Xte_s, dtype=torch.float32)).numpy().squeeze()
        p = precision_score(yte, (preds >= 0.5).astype(int))
        r = recall_score(yte, (preds >= 0.5).astype(int))
        f = f1_score(yte, (preds >= 0.5).astype(int))
        print(f'  F1={f:.4f} P={p:.4f} R={r:.4f}')

        models.append(model)
        scalers.append(scaler)
        fold_f1s.append(f)

    print(f'\n=== CV F1: {np.mean(fold_f1s):.4f} +/- {np.std(fold_f1s):.4f} ===')

In [ ]:
# Cell 9: Save results
import json

best_idx = int(np.argmax(fold_f1s))
torch.save(models[best_idx].state_dict(), f'{BASE}/scale500_fusion_model.pt')
print(f'Saved model: {BASE}/scale500_fusion_model.pt')

results = {
    'n_videos': len(set(vids_all)),
    'n_segments': int(len(y_all)),
    'positive_rate': float(y_all.mean()),
    'cross_val_f1': float(np.mean(fold_f1s)),
    'cross_val_std': float(np.std(fold_f1s)),
    'fold_f1s': [float(f) for f in fold_f1s],
}
with open(f'{SCALE_DIR}/results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved: {SCALE_DIR}/results.json')
print(json.dumps(results, indent=2))